# Draganov-contextual-mBERT: per-(lang, term) color topology

End-to-end runner and analysis for the Draganov-contextual-mBERT pipeline.
Calls the three library modules (`pointclouds`, `diagrams`, `pd_distances`) to
populate caches (no-op if already populated), then produces figures and CSVs.

**Central question:** Do contextualized mBERT embeddings of color terms form
topologically distinguishable point clouds across English, Russian, and Spanish?
The centerpiece test: does the obligatory Russian синий/голубой split surface
in the 34×34 PD-distance grid?

**Scope:** COLOR domain only — COSI 115a May 2026 analysis. 34 cells:
11 (en) + 12 (ru) + 11 (es).

**Inputs (cached by inu.1–inu.3):**
- `data/phase3/draganov_pointclouds/` — per-(lang, term) point clouds
- `data/phase3/draganov_diagrams/` — per-(lang, term) persistence diagrams
- `data/phase3/draganov_pd_distances/` — 34×34 PD-distance matrices (computed here if missing)

**Outputs:**
- `results/figures/draganov_replication_heatmap_grid.{pdf,png}`
- `results/figures/draganov_replication_russian_blues_zoom.{pdf,png}`
- `results/figures/draganov_replication_distance_consistency.{pdf,png}`
- `results/draganov_replication/per_term_cross_language_ranking.csv`
- `results/draganov_replication/russian_blues_distances.csv`

**Reference:** Draganov & Skiena (2024), Findings of EMNLP 2024.
Bespoke analysis — Draganov's C-step (phylogenetic tree over 81 Indo-European
languages) does not apply at N=3 languages. See `bd show ph-project-inu.4`.

## Setup

In [1]:
# Ensure the repo root is on sys.path (mirrors phase3_diagram_distances.ipynb pattern)
import sys
import pathlib

_NOTEBOOK_DIR = pathlib.Path('__file__').resolve().parent if '__file__' in dir() else pathlib.Path('.').resolve()
_REPO_ROOT = _NOTEBOOK_DIR.parent.parent if _NOTEBOOK_DIR.name == 'notebooks' else _NOTEBOOK_DIR

if str(_REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(_REPO_ROOT))

print(f'Repo root: {_REPO_ROOT}')

Repo root: /home/anna/ph-project-inu.4-analysis-notebook


In [2]:
import warnings

import matplotlib
matplotlib.use('Agg')  # non-interactive backend for headless execution
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning,
                        module='draganov_replication.pointclouds')

from draganov_replication.pointclouds import build_pointclouds
from draganov_replication.diagrams import compute_diagrams
from draganov_replication.pd_distances import compute_pd_distance_grid

# ── Constants ──────────────────────────────────────────────────────────────
REPO_ROOT       = _REPO_ROOT
RESULTS_DIR     = REPO_ROOT / 'results'
FIGURES_DIR     = RESULTS_DIR / 'figures'
DR_DIR          = RESULTS_DIR / 'draganov_replication'  # DR = draganov_replication

PC_DIR          = REPO_ROOT / 'data' / 'phase3' / 'draganov_pointclouds'
DG_DIR          = REPO_ROOT / 'data' / 'phase3' / 'draganov_diagrams'
PD_DIST_DIR     = REPO_ROOT / 'data' / 'phase3' / 'draganov_pd_distances'

# ── Scope ──────────────────────────────────────────────────────────────────
LANGUAGES   = ['en', 'ru', 'es']
LANG_LABELS = {'en': 'English', 'ru': 'Russian', 'es': 'Spanish'}
DOMAIN      = 'color'
DISTANCES   = ('bottleneck', 'sliced_wasserstein', 'persistence_image', 'bars_statistics')
DIMS        = (0, 1)
DIM_LABELS  = {0: 'H₀', 1: 'H₁'}
DIST_LABELS = {
    'bottleneck':         'Bottleneck',
    'sliced_wasserstein': 'Sliced Wasserstein',
    'persistence_image':  'Persistence Image',
    'bars_statistics':    'Bars Statistics',
}

SEED = 42

# ── Ensure output directories exist ────────────────────────────────────────
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
DR_DIR.mkdir(parents=True, exist_ok=True)

print('Setup complete.')
print(f'Repo root  : {REPO_ROOT}')
print(f'Figures dir: {FIGURES_DIR}')
print(f'Output dir : {DR_DIR}')

Setup complete.
Repo root  : /home/anna/ph-project-inu.4-analysis-notebook
Figures dir: /home/anna/ph-project-inu.4-analysis-notebook/results/figures
Output dir : /home/anna/ph-project-inu.4-analysis-notebook/results/draganov_replication


In [3]:
def save_fig(fig, stem: str) -> None:
    """Export a figure as both .pdf (vector) and .png (300 dpi) to FIGURES_DIR.
    Verbatim from phase3_diagram_distances.ipynb save_fig convention.
    """
    pdf_path = FIGURES_DIR / f'{stem}.pdf'
    png_path = FIGURES_DIR / f'{stem}.png'
    fig.savefig(pdf_path, bbox_inches='tight')
    fig.savefig(png_path, dpi=300, bbox_inches='tight')
    print(f'  Saved {pdf_path.name}  +  {png_path.name}')

print('save_fig helper defined.')

save_fig helper defined.


## Pipeline run (idempotent)

Calls the three library modules in sequence. Each is idempotent: if cached
outputs already exist they are loaded from disk rather than recomputed.

- `build_pointclouds()`: 34 per-(lang, term) point clouds from mBERT embeddings
- `compute_diagrams()`: 34 persistence diagrams (H₀ + H₁) via ripser
- `compute_pd_distance_grid()`: 8 distance matrices (4 distances × 2 dims), 34×34 each

In [4]:
import logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')

print('Building point clouds (no-op if cached) ...')
pc_manifest = build_pointclouds(
    emb_dir=REPO_ROOT / 'data' / 'phase3' / 'embeddings',
    canon_dir=REPO_ROOT / 'canon-terms',
    out_dir=PC_DIR,
)
print(f'Point-cloud manifest: {len(pc_manifest)} cells')
print()

print('Computing persistence diagrams (no-op if cached) ...')
dg_manifest = compute_diagrams(
    pointcloud_dir=PC_DIR,
    out_dir=DG_DIR,
)
print(f'Diagram manifest: {len(dg_manifest)} cells')
print()

print('Computing PD distance grids (no-op if cached) ...')
grids = compute_pd_distance_grid(
    diagrams_dir=DG_DIR,
    out_dir=PD_DIST_DIR,
)
print(f'Distance grids: {len(grids)} matrices')
for (dist, dim), M in sorted(grids.items()):
    print(f'  ({dist}, H_{dim}): shape={M.shape}, max={M.max():.4f}')

INFO draganov_replication.pointclouds: Manifest written: /home/anna/ph-project-inu.4-analysis-notebook/data/phase3/draganov_pointclouds/manifest.csv (34 rows)


INFO draganov_replication.diagrams: Manifest written: /home/anna/ph-project-inu.4-analysis-notebook/data/phase3/draganov_diagrams/manifest.csv (34 rows)


INFO draganov_replication.pd_distances: Loading from cache: /home/anna/ph-project-inu.4-analysis-notebook/data/phase3/draganov_pd_distances/bottleneck_d0.npz


INFO draganov_replication.pd_distances: Loading from cache: /home/anna/ph-project-inu.4-analysis-notebook/data/phase3/draganov_pd_distances/bottleneck_d1.npz


INFO draganov_replication.pd_distances: Loading from cache: /home/anna/ph-project-inu.4-analysis-notebook/data/phase3/draganov_pd_distances/sliced_wasserstein_d0.npz


...

INFO draganov_replication.pd_distances: Loading from cache: /home/anna/ph-project-inu.4-analysis-notebook/data/phase3/draganov_pd_distances/persistence_image_d0.npz


INFO draganov_replication.pd_distances: Loading from cache: /home/anna/ph-project-inu.4-analysis-notebook/data/phase3/draganov_pd_distances/persistence_image_d1.npz


INFO draganov_replication.pd_distances: Loading from cache: /home/anna/ph-project-inu.4-analysis-notebook/data/phase3/draganov_pd_distances/bars_statistics_d0.npz


INFO draganov_replication.pd_distances: Loading from cache: /home/anna/ph-project-inu.4-analysis-notebook/data/phase3/draganov_pd_distances/bars_statistics_d1.npz


Building point clouds (no-op if cached) ...
Point-cloud manifest: 34 cells

Computing persistence diagrams (no-op if cached) ...
Diagram manifest: 34 cells

Computing PD distance grids (no-op if cached) ...
Distance grids: 8 matrices
  (bars_statistics, H_0): shape=(34, 34), max=0.7677
  (bars_statistics, H_1): shape=(34, 34), max=2.4137
  (bottleneck, H_0): shape=(34, 34), max=0.2417
  (bottleneck, H_1): shape=(34, 34), max=0.0823
  (persistence_image, H_0): shape=(34, 34), max=2.6193
  (persistence_image, H_1): shape=(34, 34), max=0.1601
  (sliced_wasserstein, H_0): shape=(34, 34), max=20.7224
  (sliced_wasserstein, H_1): shape=(34, 34), max=1.6039


## Cell labels and language block indices

Load the sorted cell list from the diagrams manifest. Cells are sorted by
`(lang, term)` — the same order as the distance matrices. Compute the block
boundaries for the en | es | ru divider lines in heatmaps.

In [5]:
# Load sorted manifest (same sort order as compute_pd_distance_grid)
dg_manifest_sorted = pd.read_csv(DG_DIR / 'manifest.csv')
dg_manifest_sorted = dg_manifest_sorted.sort_values(['lang', 'term']).reset_index(drop=True)

cells = [(row['lang'], row['term']) for _, row in dg_manifest_sorted.iterrows()]
cell_labels = [f"{lang}/{term}" for lang, term in cells]
n_cells = len(cells)

# Language block sizes (sorted: en, es, ru — alphabetical)
lang_order = sorted(LANGUAGES)  # ['en', 'es', 'ru']
lang_counts = {lang: sum(1 for l, _ in cells if l == lang) for lang in lang_order}
lang_boundaries = []  # x (or y) positions where divider lines go between blocks
cumsum = 0
for lang in lang_order[:-1]:
    cumsum += lang_counts[lang]
    lang_boundaries.append(cumsum - 0.5)

# Short tick labels: just the term (not lang/term) — lang shown by dividers
tick_labels = [term for _, term in cells]

print(f'Total cells: {n_cells}')
print(f'Language order: {lang_order}')
print(f'Language counts: {lang_counts}')
print(f'Divider positions: {lang_boundaries}')
print(f'First few cells: {cell_labels[:5]}')

Total cells: 34
Language order: ['en', 'es', 'ru']
Language counts: {'en': 11, 'es': 11, 'ru': 12}
Divider positions: [10.5, 21.5]
First few cells: ['en/black', 'en/blue', 'en/brown', 'en/gray', 'en/green']


## Figure 1 — 34×34 heatmap grid

4×2 grid of heatmaps (4 distances × 2 homology dimensions). Each panel shows
the 34×34 PD-distance matrix sorted by `(lang, term)`. Faint vertical and
horizontal lines separate the three language blocks (en | es | ru, alphabetical).
Per-panel colourbars reflect that the four distances have very different
absolute scales.

Saved to `results/figures/draganov_replication_heatmap_grid.{pdf,png}`.

In [6]:
fig, axes = plt.subplots(
    nrows=len(DISTANCES),
    ncols=len(DIMS),
    figsize=(16, 28),
)

for row_i, distance in enumerate(DISTANCES):
    for col_j, dim in enumerate(DIMS):
        ax = axes[row_i, col_j]
        M = grids[(distance, dim)]

        im = ax.imshow(
            M,
            cmap='viridis',
            aspect='auto',
            interpolation='nearest',
        )

        # Language block divider lines
        for bp in lang_boundaries:
            ax.axhline(bp, color='white', linewidth=0.8, alpha=0.7)
            ax.axvline(bp, color='white', linewidth=0.8, alpha=0.7)

        # Per-panel colorbar
        cbar = fig.colorbar(im, ax=ax, shrink=0.85, pad=0.02)
        cbar.set_label('Distance', fontsize=8)
        cbar.ax.tick_params(labelsize=7)

        ax.set_title(
            f'{DIST_LABELS[distance]} — {DIM_LABELS[dim]}',
            fontsize=10, pad=4,
        )

        # Tick labels: one per cell, rotated for legibility
        ax.set_xticks(range(n_cells))
        ax.set_yticks(range(n_cells))
        ax.set_xticklabels(tick_labels, rotation=90, fontsize=5)
        ax.set_yticklabels(tick_labels, fontsize=5)

        # Language block labels on the left of the first column
        if col_j == 0:
            cumpos = 0
            for lang in lang_order:
                cnt = lang_counts[lang]
                mid = cumpos + cnt / 2 - 0.5
                ax.annotate(
                    LANG_LABELS[lang],
                    xy=(-0.25, mid),
                    xycoords=('axes fraction', 'data'),
                    ha='center', va='center',
                    fontsize=8, rotation=90, color='navy',
                    annotation_clip=False,
                )
                cumpos += cnt

fig.suptitle(
    '34×34 PD-distance heatmaps — Draganov-contextual-mBERT pipeline\n'
    'Color domain (EN / ES / RU, sorted alphabetically by lang then term)\n'
    'White lines separate language blocks; per-panel colourbars',
    fontsize=12, y=1.005,
)

plt.tight_layout()
save_fig(fig, 'draganov_replication_heatmap_grid')
plt.close(fig)

  Saved draganov_replication_heatmap_grid.pdf  +  draganov_replication_heatmap_grid.png


## Figure 2 — Russian-blues zoom

Centerpiece test: does the obligatory Russian синий (dark blue) / голубой (light blue)
split surface as elevated distances in the PD-distance grid?

The zoom extracts the 4×4 submatrix over the cells
`{en/blue, es/azul, ru/синий, ru/голубой}` from each of the 8 distance matrices,
producing a 4×2 grid of annotated heatmaps.

**Predicted pattern (Paramei 2005, Winawer et al. 2007):**
If the синий/голубой split is encoded in mBERT's contextual embedding topology,
then `d(ru/синий, ru/голубой)` should be *elevated* relative to same-language pairs,
and `d(en/blue, ru/синий)` may differ from `d(en/blue, ru/голубой)` (and similarly
for `es/azul`).

Saved to `results/figures/draganov_replication_russian_blues_zoom.{pdf,png}`.

In [7]:
# ── Identify zoom cells ─────────────────────────────────────────────
ZOOM_CELLS = [
    ('en', 'blue'),
    ('es', 'azul'),
    ('ru', 'синий'),
    ('ru', 'голубой'),
]
ZOOM_LABELS = ['en/blue', 'es/azul', 'ru/синий\n(dark)', 'ru/голубой\n(light)']

zoom_indices = []
for target in ZOOM_CELLS:
    idx = next(
        (i for i, c in enumerate(cells) if c == target),
        None,
    )
    if idx is None:
        raise ValueError(f'Zoom cell {target} not found in cells list')
    zoom_indices.append(idx)

print(f'Zoom cell indices: {list(zip(ZOOM_CELLS, zoom_indices))}')

fig, axes = plt.subplots(
    nrows=len(DISTANCES),
    ncols=len(DIMS),
    figsize=(11, 20),
)

for row_i, distance in enumerate(DISTANCES):
    for col_j, dim in enumerate(DIMS):
        ax = axes[row_i, col_j]
        M_full = grids[(distance, dim)]

        # Extract 4x4 submatrix
        M_zoom = M_full[np.ix_(zoom_indices, zoom_indices)]

        im = ax.imshow(
            M_zoom,
            cmap='YlOrRd',
            aspect='auto',
            interpolation='nearest',
        )

        # Annotate each cell with its distance value
        for i in range(len(ZOOM_CELLS)):
            for j in range(len(ZOOM_CELLS)):
                val = M_zoom[i, j]
                text_color = 'white' if val > 0.6 * M_zoom.max() else 'black'
                ax.text(
                    j, i, f'{val:.3f}',
                    ha='center', va='center',
                    fontsize=8, color=text_color, fontweight='bold',
                )

        # Highlight the sinij-goluboy pair with a box
        sinij_idx_local = 2    # ru/sinij is index 2 in zoom
        goluboy_idx_local = 3  # ru/goluboy is index 3 in zoom
        for (ri, ci) in [
            (sinij_idx_local, goluboy_idx_local),
            (goluboy_idx_local, sinij_idx_local),
        ]:
            rect = mpatches.Rectangle(
                (ci - 0.5, ri - 0.5), 1, 1,
                linewidth=2, edgecolor='blue', facecolor='none',
            )
            ax.add_patch(rect)

        cbar = fig.colorbar(im, ax=ax, shrink=0.8, pad=0.02)
        cbar.ax.tick_params(labelsize=7)

        ax.set_xticks(range(len(ZOOM_CELLS)))
        ax.set_yticks(range(len(ZOOM_CELLS)))
        ax.set_xticklabels(ZOOM_LABELS, fontsize=8)
        ax.set_yticklabels(ZOOM_LABELS, fontsize=8)
        ax.set_title(
            f'{DIST_LABELS[distance]} — {DIM_LABELS[dim]}',
            fontsize=9, pad=4,
        )

fig.suptitle(
    'Russian-blues zoom: {en/blue, es/azul, ru/синий, ru/голубой}\n'
    'Blue boxes highlight the синий–голубой within-Russian pair\n'
    'Annotated values are PD distances; all 8 (distance, dim) combinations shown',
    fontsize=11, y=1.01,
)

plt.tight_layout()
save_fig(fig, 'draganov_replication_russian_blues_zoom')
plt.close(fig)

Zoom cell indices: [(('en', 'blue'), 1), (('es', 'azul'), 12), (('ru', 'синий'), 31), (('ru', 'голубой'), 23)]


  Saved draganov_replication_russian_blues_zoom.pdf  +  draganov_replication_russian_blues_zoom.png


## Figure 3 — Cross-distance consistency

Do the four PD-distance notions agree on which pairs of (lang, term) cells are
close vs. distant?

For each of the 8 matrices (4 distances × 2 dims), take the upper triangle
(34 choose 2 = 561 values) and compute the Spearman rank correlation between
every pair of upper triangles. The resulting 8×8 correlation heatmap shows how
consistent the distance notions are in their ordering of the 561 cell-pairs.

High correlation (≈ 1.0) between two (distance, dim) combinations means they
agree on which pairs are topologically similar — a sign of robustness. Low
correlation means the choice of metric matters.

Saved to `results/figures/draganov_replication_distance_consistency.{pdf,png}`.

In [8]:
# ── Build 8x8 Spearman correlation matrix ──────────────────────────
# Key order: (distance, dim) pairs sorted consistently
keys_ordered = [(dist, dim) for dist in DISTANCES for dim in DIMS]
key_labels   = [f'{DIST_LABELS[d][:6]}\n{DIM_LABELS[dim]}' for d, dim in keys_ordered]

# Upper-triangle indices (excluding diagonal)
tri_i, tri_j = np.triu_indices(n_cells, k=1)  # 561 pairs

# Build matrix of upper-triangle vectors: shape (8, 561)
uppers = np.stack(
    [grids[key][tri_i, tri_j].astype(np.float64) for key in keys_ordered],
    axis=0,
)

n_combos = len(keys_ordered)
spearman_mat = np.ones((n_combos, n_combos), dtype=np.float64)

for i in range(n_combos):
    for j in range(i + 1, n_combos):
        rho, _ = spearmanr(uppers[i], uppers[j])
        spearman_mat[i, j] = rho
        spearman_mat[j, i] = rho

print('Spearman rank correlation matrix (upper triangles of 34x34 distance grids):')
df_spearman = pd.DataFrame(
    spearman_mat,
    index=[f'{d[:4]}/{DIM_LABELS[dim]}' for d, dim in keys_ordered],
    columns=[f'{d[:4]}/{DIM_LABELS[dim]}' for d, dim in keys_ordered],
)
print(df_spearman.to_string(float_format='{:.3f}'.format))

# ── Figure ──────────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 8))

im = ax.imshow(
    spearman_mat,
    cmap='RdBu_r',
    vmin=-1, vmax=1,
    aspect='auto',
    interpolation='nearest',
)

# Annotate each cell
for i in range(n_combos):
    for j in range(n_combos):
        text_color = 'white' if abs(spearman_mat[i, j]) > 0.6 else 'black'
        ax.text(
            j, i, f'{spearman_mat[i, j]:.2f}',
            ha='center', va='center',
            fontsize=8, color=text_color,
        )

cbar = fig.colorbar(im, ax=ax, shrink=0.85)
cbar.set_label('Spearman ρ', fontsize=11)

ax.set_xticks(range(n_combos))
ax.set_yticks(range(n_combos))
ax.set_xticklabels(key_labels, fontsize=8)
ax.set_yticklabels(key_labels, fontsize=8)
ax.set_title(
    'Cross-distance consistency: Spearman ρ between upper triangles\n'
    '(561 pairwise distances per (distance, dim) combination)\n'
    'High ρ = distance notions agree on cell-pair ordering',
    fontsize=11,
)

plt.tight_layout()
save_fig(fig, 'draganov_replication_distance_consistency')
plt.close(fig)

Spearman rank correlation matrix (upper triangles of 34x34 distance grids):
         bott/H₀  bott/H₁  slic/H₀  slic/H₁  pers/H₀  pers/H₁  bars/H₀  bars/H₁
bott/H₀    1.000    0.101    0.816    0.304    0.795    0.384    0.848    0.555
bott/H₁    0.101    1.000    0.105    0.315    0.118    0.273    0.022    0.122
slic/H₀    0.816    0.105    1.000    0.475    0.883    0.464    0.837    0.586
slic/H₁    0.304    0.315    0.475    1.000    0.341    0.562    0.295    0.247
pers/H₀    0.795    0.118    0.883    0.341    1.000    0.475    0.722    0.577
pers/H₁    0.384    0.273    0.464    0.562    0.475    1.000    0.405    0.559
bars/H₀    0.848    0.022    0.837    0.295    0.722    0.405    1.000    0.677
bars/H₁    0.555    0.122    0.586    0.247    0.577    0.559    0.677    1.000


  Saved draganov_replication_distance_consistency.pdf  +  draganov_replication_distance_consistency.png


## Table 1 — Per-term cross-language ranking

For each (lang, term) cell and each (distance, dim) combination, rank the closest
cells in *other* languages by distance. The top 5 closest cross-language neighbours
per cell are saved to `results/draganov_replication/per_term_cross_language_ranking.csv`.

- 34 source cells × 8 (distance, dim) combinations × top-5 = 1360 rows
- Columns: `[lang, term, distance, dim, rank, target_lang, target_term, distance_value]`

This table answers: which color terms across languages are topologically nearest
neighbours in persistence-diagram distance space?

In [9]:
TOP_K = 5
ranking_rows = []

for (distance, dim), M in grids.items():
    for src_idx, (src_lang, src_term) in enumerate(cells):
        row_distances = M[src_idx]  # distances to all 34 cells

        # Candidate indices: only other-language cells
        cand_indices = [
            idx for idx, (tgt_lang, _) in enumerate(cells)
            if tgt_lang != src_lang
        ]

        # Sort candidates by distance, take top-5
        cand_dists = [(row_distances[idx], idx) for idx in cand_indices]
        cand_dists.sort(key=lambda x: x[0])
        top5 = cand_dists[:TOP_K]

        for rank, (dist_val, tgt_idx) in enumerate(top5, start=1):
            tgt_lang, tgt_term = cells[tgt_idx]
            ranking_rows.append({
                'lang':           src_lang,
                'term':           src_term,
                'distance':       distance,
                'dim':            dim,
                'rank':           rank,
                'target_lang':    tgt_lang,
                'target_term':    tgt_term,
                'distance_value': float(dist_val),
            })

ranking_df = pd.DataFrame(
    ranking_rows,
    columns=['lang', 'term', 'distance', 'dim', 'rank', 'target_lang', 'target_term', 'distance_value'],
)

assert len(ranking_df) == 1360, f'Expected 1360 rows, got {len(ranking_df)}'
assert (ranking_df['lang'] != ranking_df['target_lang']).all(), \
    "ranking must exclude same-language cells"

ranking_csv_path = DR_DIR / 'per_term_cross_language_ranking.csv'
ranking_df.to_csv(ranking_csv_path, index=False)
print(f'Written: {ranking_csv_path}  ({len(ranking_df)} rows)')
print('\nSample rows (en/blue, bottleneck, H₁):')
sample = ranking_df[
    (ranking_df['lang'] == 'en') &
    (ranking_df['term'] == 'blue') &
    (ranking_df['distance'] == 'bottleneck') &
    (ranking_df['dim'] == 1)
]
print(sample.to_string(index=False))

Written: /home/anna/ph-project-inu.4-analysis-notebook/results/draganov_replication/per_term_cross_language_ranking.csv  (1360 rows)

Sample rows (en/blue, bottleneck, H₁):
lang term   distance  dim  rank target_lang target_term  distance_value
  en blue bottleneck    1     1          ru     зелёный        0.026491
  en blue bottleneck    1     2          es       negro        0.028967
  en blue bottleneck    1     3          ru      чёрный        0.029019
  en blue bottleneck    1     4          es       verde        0.029118
  en blue bottleneck    1     5          es      blanco        0.030292


## Table 2 — Russian-blues distances

The 6 unique pairs from the 4-cell Russian-blues subset
`{en/blue, es/azul, ru/синий, ru/голубой}` across all 8 (distance, dim) combinations.

Concretely verifies the centerpiece signal: all values must be finite and > 0.

Saved to `results/draganov_replication/russian_blues_distances.csv`.

In [10]:
from itertools import combinations

# ── Build the 6-pair x 8-combo table ─────────────────────────────
blues_pairs = list(combinations(range(len(ZOOM_CELLS)), 2))  # 6 unique pairs

blues_rows = []
for i, j in blues_pairs:
    pair_label = (
        f'{ZOOM_CELLS[i][0]}/{ZOOM_CELLS[i][1]} '
        f'— '
        f'{ZOOM_CELLS[j][0]}/{ZOOM_CELLS[j][1]}'
    )
    row = {'pair': pair_label}
    for distance in DISTANCES:
        for dim in DIMS:
            M_full = grids[(distance, dim)]
            val = float(M_full[zoom_indices[i], zoom_indices[j]])
            row[f'{distance}_d{dim}'] = val
    blues_rows.append(row)

blues_df = pd.DataFrame(blues_rows)

# ── Acceptance assertions ──────────────────────────────────────────────
assert len(blues_df) == 6, f'Expected 6 rows, got {len(blues_df)}'

value_cols = [c for c in blues_df.columns if c != 'pair']
assert len(value_cols) == 8, f'Expected 8 value columns, got {len(value_cols)}'

# All values finite and > 0
vals = blues_df[value_cols].values
assert np.all(np.isfinite(vals)), 'Non-finite values found in russian_blues_distances'
assert np.all(vals > 0), (
    f'Non-positive values found in russian_blues_distances:\n'
    f'{blues_df[value_cols][~(blues_df[value_cols] > 0).all(axis=1)]}'
)

blues_csv_path = DR_DIR / 'russian_blues_distances.csv'
blues_df.to_csv(blues_csv_path, index=False)
print(f'Written: {blues_csv_path}  ({len(blues_df)} rows x {len(value_cols)} distance columns)')
print('All values finite and > 0: PASS')
print()
print(blues_df.to_string(index=False))

Written: /home/anna/ph-project-inu.4-analysis-notebook/results/draganov_replication/russian_blues_distances.csv  (6 rows x 8 distance columns)
All values finite and > 0: PASS

                 pair  bottleneck_d0  bottleneck_d1  sliced_wasserstein_d0  sliced_wasserstein_d1  persistence_image_d0  persistence_image_d1  bars_statistics_d0  bars_statistics_d1
    en/blue — es/azul       0.129471       0.052802              11.211836               0.924252              1.139181              0.059928            0.219849            0.744779
   en/blue — ru/синий       0.096895       0.042091               8.916497               0.754717              0.874944              0.042263            0.173285            0.603161
 en/blue — ru/голубой       0.088434       0.039306               7.597063               0.714202              0.663751              0.018931            0.184119            0.489669
   es/azul — ru/синий       0.096098       0.029415               5.340193               0.74791

## Commentary: Interpreting the results

### Russian-blues split

The centerpiece test asks whether the obligatory Russian синий/голубой distinction —
which Paramei (2005) and Winawer et al. (2007) show to have perceptual consequences —
is visible in the topology of contextualized mBERT embeddings.

The `russian_blues_distances.csv` table gives the concrete answer: 6 pairwise distances
between `{en/blue, es/azul, ru/синий, ru/голубой}` for each of the 8 (distance, dim)
combinations. The key comparisons are:

- **`ru/синий — ru/голубой` (within-Russian):** Is this distance elevated relative to other
  within-language pairs? A positive answer would mean that mBERT's embedding topology
  tracks the categorical синий/голубой boundary, not just the label difference.
- **`en/blue — ru/синий` vs. `en/blue — ru/голубой`:** Does English *blue* align more
  strongly with синий (dark blue) or голубой (light blue)? A preference for синий would
  be consistent with cross-linguistic evidence that English *blue* is roughly coextensive
  with Russian синий (Lindsey & Brown 2009).
- **`es/azul — ru/синий` vs. `es/azul — ru/голубой`:** Symmetric question for Spanish *azul*.

The Figure 2 heatmaps (Russian-blues zoom) and the `russian_blues_distances.csv` table
provide the quantitative evidence for these comparisons. Annotated distance values are
readable directly from the heatmap panels.

### H₀ vs H₁

The two homology dimensions capture structurally different aspects of the persistence
diagram:

- **H₀ (connected components):** How many distinct clusters form in the point cloud,
  and at what scales do they merge? H₀ bars track the birth and death of connected
  components as the Vietoris-Rips filtration grows. For mBERT embeddings, H₀ structure
  reflects how modular or fragmented the representation cloud is — whether the color-term
  contexts form a single tightly-knit group or several loosely connected subclusters.
- **H₁ (1-cycles / loops):** Which loops close in the point cloud, and at what scales?
  H₁ bars correspond to topological holes — configurations where the data points
  surround an empty region. In embedding space, a H₁ loop typically signals a
  "ring-like" distribution of contexts that avoids a central void, which can reflect
  semantic subdivision within the term's context distribution.

Comparing the consistency matrix (Figure 3) between H₀ and H₁ rows reveals whether
the two dimensions agree on the same ordering of pairwise distances. High cross-dimension
Spearman ρ would mean the topology is dominated by connected-component structure (the H₁
loops are just echoing the H₀ geometry). Low ρ across dimensions would indicate that
the two levels of topological structure capture meaningfully different aspects of the
embedding space.

### Cross-distance consistency (Figure 3)

The 8×8 Spearman rank correlation matrix answers: do the four distance notions agree on
which cell-pairs are close vs. distant? The interpretive key:

- **Bottleneck distance** is determined entirely by the single largest unmatched feature.
  It is sensitive to outlier persistence bars and insensitive to the accumulation of
  small features.
- **Sliced Wasserstein** integrates over all features (with M=50 directions), weighting
  by persistence. It is more sensitive to the overall shape of the diagram than bottleneck.
- **Persistence image (L2)** vectorises the diagram into a 2D density image and compares
  images by Euclidean distance. Smoothing (σ=0.1) means nearby features contribute
  jointly.
- **Bars statistics (L2)** vectorises each diagram into a 10- or 40-dimensional statistics
  vector (mean, std, median, IQR, range, percentiles, entropy) and compares by L2. It is
  sensitive to moment-level differences but blind to fine-grained topological structure.

High Spearman ρ across all four distances would support the robustness claim: the
cross-linguistic topology differences are not an artifact of the metric choice.

### Caveats: undertarget cells

Three of the 34 cells fall below the n ≥ 200 target:

- **ru/фиолетовый** (n = 104): Under-target after the multi-year news rescue (i6q).
  Kept as a documented casualty — it is a Berlin & Kay basic color term and n = 104
  is sufficient to compute a persistence diagram, but the resulting diagram may be
  noisier than the n ≥ 199 terms. Tracked in `bd show ph-project-bcy`.
- **ru/коричневый** (n = 163): Moderate shortfall. Diagram computable; distances
  involving this cell will be slightly noisier.
- **es/marrón** (n = 161): Same situation as ru/коричневый.

All four cells in the Russian-blues centerpiece comparison (en/blue, es/azul,
ru/синий, ru/голубой) are at n ≥ 198, so the centerpiece test is unaffected.
Grid distances *to or from* the three undertarget cells should be interpreted
with additional caution.

### Note on permutation testing

This analysis does not include permutation tests on PD distances. The Kushnareva
pipeline (`replication/diagram_distances.py`) supports permutation tests because
each (lang, term) cell has a *distribution* of persistence diagrams (one per sentence).
Here, each cell has a *single* persistence diagram (the topology of the full n-point
cloud). Single-PD permutation tests would require a different framing — permuting
the cell assignments themselves — which is a separate analysis. See the inu epic for
discussion of why this design choice was made and how a permutation framing could be
added as a follow-up.